In [0]:
# =============================================================================
# Phase 3 — Hive Data Warehouse
# Craigslist Used Vehicle Dataset
# Registers the Featured Dataset as a Hive External Table,
# validates with SQL queries, and exports a single CSV for Tableau.
# =============================================================================

import logging
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleMarket_Hive")

In [0]:
# =============================================================================
# CONFIG
# =============================================================================

FEATURED_TABLE = "workspace.default.featured_vehicles"
EXPORT_VOLUME  = "/Volumes/workspace/default/vehicle_data/vehicles_featured_tableau"

# Keep these for SQL query compatibility — they map to the Delta table
HIVE_DATABASE  = "workspace.default"
HIVE_TABLE     = "featured_vehicles"

In [0]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_Hive") \
    .getOrCreate()

pipeline_start = time.time()
log.info("=" * 65)
log.info("  Phase 5 — Delta Warehouse")
log.info("=" * 65)

2026-07-26 18:44:50 | INFO | =================================================================
2026-07-26 18:44:50 | INFO |   Phase 5 — Delta Warehouse
2026-07-26 18:44:50 | INFO | =================================================================


In [0]:
# =============================================================================
# 2. LOAD FEATURED DELTA TABLE
# =============================================================================

log.info("Loading Featured Delta table...")

df = spark.read.table(FEATURED_TABLE)

record_count_in = df.count()
columns_in      = len(df.columns)

log.info(f"  Records loaded  : {record_count_in}")
log.info(f"  Columns loaded  : {columns_in}")
log.info("  Schema:")
df.printSchema()

log.info("  Sample rows:")
df.show(5, truncate=False)

2026-07-26 18:44:50 | INFO | Loading Featured Delta table...
2026-07-26 18:44:52 | INFO |   Records loaded  : 244772
2026-07-26 18:44:52 | INFO |   Columns loaded  : 49
2026-07-26 18:44:52 | INFO |   Schema:
2026-07-26 18:44:52 | INFO |   Sample rows:


root
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- posting_date: timestamp (nullable = true)
 |-- vehicle_age:

In [0]:
# =============================================================================
# 3. VERIFY DELTA TABLE IS ACCESSIBLE
# =============================================================================

log.info(f"Verifying Delta table '{FEATURED_TABLE}'...")

spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

log.info("  Table listing complete.")

2026-07-26 18:44:54 | INFO | Verifying Delta table 'workspace.default.featured_vehicles'...
2026-07-26 18:44:54 | INFO |   Table listing complete.


+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|default |bronze_vehicles   |false      |
|default |featured_vehicles |false      |
|default |silver_vehicles   |false      |
|default |vehicles_validated|false      |
+--------+------------------+-----------+



In [0]:
# =============================================================================
# 4. FEATURED DELTA TABLE — NO DDL REQUIRED
# =============================================================================

log.info(f"Featured Delta table '{FEATURED_TABLE}' is a managed Delta table.")
log.info("  No CREATE TABLE required — registered via saveAsTable in notebook 02.")
df.printSchema()

2026-07-26 18:44:55 | INFO | Featured Delta table 'workspace.default.featured_vehicles' is a managed Delta table.
2026-07-26 18:44:55 | INFO |   No CREATE TABLE required — registered via saveAsTable in notebook 02.


root
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- posting_date: timestamp (nullable = true)
 |-- vehicle_age:

In [0]:
# =============================================================================
# 5. VALIDATE TABLE REGISTRATION
# Purpose : Confirms the database, table, and schema are correctly
#           registered in Hive before proceeding to analytical queries.
# =============================================================================

log.info("Validating Hive table registration...")

print("\n" + "=" * 65)
print(f"  SHOW TABLES IN {HIVE_DATABASE}")
print("=" * 65)
spark.sql(f"SHOW TABLES IN {HIVE_DATABASE}").show(truncate=False)

print("\n" + "=" * 65)
print(f"  DESCRIBE {HIVE_DATABASE}.{HIVE_TABLE}")
print("=" * 65)
spark.sql(f"DESCRIBE {HIVE_DATABASE}.{HIVE_TABLE}").show(200, truncate=False)

print("\n" + "=" * 65)
print(f"  SELECT * FROM {HIVE_DATABASE}.{HIVE_TABLE} LIMIT 10")
print("=" * 65)
spark.sql(f"SELECT * FROM {HIVE_DATABASE}.{HIVE_TABLE} LIMIT 10").show(truncate=False)

log.info("  Table registration validated.")

2026-07-26 18:44:55 | INFO | Validating Hive table registration...



  SHOW TABLES IN workspace.default
+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|default |bronze_vehicles   |false      |
|default |featured_vehicles |false      |
|default |silver_vehicles   |false      |
|default |vehicles_validated|false      |
+--------+------------------+-----------+


  DESCRIBE workspace.default.featured_vehicles
+--------------------+---------+-------+
|col_name            |data_type|comment|
+--------------------+---------+-------+
|manufacturer        |string   |NULL   |
|model               |string   |NULL   |
|id                  |string   |NULL   |
|url                 |string   |NULL   |
|region              |string   |NULL   |
|region_url          |string   |NULL   |
|price               |int      |NULL   |
|year                |int      |NULL   |
|condition           |string   |NULL   |
|cylinders           |string   |NULL   |
|fuel                |string   |NULL   |
|odom

2026-07-26 18:44:56 | INFO |   Table registration validated.


+------------+----------------------+----------+---------------------------------------------------------------------------------------------+-------+------------------------------+-----+----+---------+-----------+------+--------+------------+------------+-----------------+-------+-------+--------+-----------+-------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# =============================================================================
# 6. ANALYTICAL SQL VALIDATION QUERIES
# Purpose : Demonstrates Hive as an analytical SQL engine.
#           These queries are not saved as tables — they validate the
#           warehouse is functioning correctly and show business insights.
# =============================================================================

log.info("Running analytical SQL validation queries...")

# --- 6a. Total record count ---
print("\n" + "=" * 65)
print("  6a. Total Record Count")
print("=" * 65)
spark.sql(f"""
    SELECT COUNT(*) AS total_records
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
""").show()

# --- 6b. Manufacturer-wise listing count (Top 15) ---
print("\n" + "=" * 65)
print("  6b. Top 15 Manufacturers by Listing Count")
print("=" * 65)
spark.sql(f"""
    SELECT manufacturer,
           COUNT(*)              AS listings,
           ROUND(AVG(price), 2)  AS avg_price
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY manufacturer
    ORDER BY listings DESC
    LIMIT 15
""").show(truncate=False)

# --- 6c. Fuel type distribution ---
print("\n" + "=" * 65)
print("  6c. Fuel Type Distribution")
print("=" * 65)
spark.sql(f"""
    SELECT fuel,
           COUNT(*)                                             AS listings,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY fuel
    ORDER BY listings DESC
""").show(truncate=False)

# --- 6d. Average vehicle price ---
print("\n" + "=" * 65)
print("  6d. Average Vehicle Price")
print("=" * 65)
spark.sql(f"""
    SELECT ROUND(AVG(price), 2)  AS avg_price,
           MIN(price)            AS min_price,
           MAX(price)            AS max_price
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
""").show()

# --- 6e. State-wise listing count (Top 15) ---
print("\n" + "=" * 65)
print("  6e. Top 15 States by Listing Count")
print("=" * 65)
spark.sql(f"""
    SELECT state,
           COUNT(*) AS listings
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY state
    ORDER BY listings DESC
    LIMIT 15
""").show(truncate=False)

# --- 6f. Condition distribution ---
print("\n" + "=" * 65)
print("  6f. Condition Distribution")
print("=" * 65)
spark.sql(f"""
    SELECT `condition`,
           COUNT(*)                                             AS listings,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY `condition`
    ORDER BY listings DESC
""").show(truncate=False)

# --- 6g. Transmission distribution ---
print("\n" + "=" * 65)
print("  6g. Transmission Distribution")
print("=" * 65)
spark.sql(f"""
    SELECT transmission,
           COUNT(*)                                             AS listings,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY transmission
    ORDER BY listings DESC
""").show(truncate=False)

# --- 6h. Average depreciation index by manufacturer (Top 15) ---
print("\n" + "=" * 65)
print("  6h. Avg Depreciation Index by Manufacturer (Top 15)")
print("=" * 65)
spark.sql(f"""
    SELECT manufacturer,
           ROUND(AVG(depreciation_index), 2) AS avg_depreciation_index,
           COUNT(*)                          AS listings
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    WHERE depreciation_index IS NOT NULL
    GROUP BY manufacturer
    ORDER BY avg_depreciation_index DESC
    LIMIT 15
""").show(truncate=False)

# --- 6i. Weekend vs Weekday listing split ---
print("\n" + "=" * 65)
print("  6i. Weekend vs Weekday Listing Split")
print("=" * 65)
spark.sql(f"""
    SELECT is_weekend,
           COUNT(*)                                             AS listings,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY is_weekend
    ORDER BY is_weekend
""").show(truncate=False)

# --- 6j. Posting period distribution ---
print("\n" + "=" * 65)
print("  6j. Posting Period Distribution")
print("=" * 65)
spark.sql(f"""
    SELECT posting_period,
           COUNT(*)                                             AS listings,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {HIVE_DATABASE}.{HIVE_TABLE}
    GROUP BY posting_period
    ORDER BY listings DESC
""").show(truncate=False)

log.info("  All analytical SQL queries completed.")

2026-07-26 18:44:57 | INFO | Running analytical SQL validation queries...



  6a. Total Record Count
+-------------+
|total_records|
+-------------+
|       244772|
+-------------+


  6b. Top 15 Manufacturers by Listing Count
+-------------+--------+---------+
|manufacturer |listings|avg_price|
+-------------+--------+---------+
|ford         |40904   |47487.72 |
|chevrolet    |32190   |21052.9  |
|toyota       |20750   |13462.86 |
|honda        |14773   |10105.0  |
|nissan       |12406   |21552.43 |
|jeep         |11370   |238781.61|
|gmc          |9028    |34829.74 |
|ram          |8802    |28347.61 |
|dodge        |8311    |13255.45 |
|unknown      |8278    |48910.71 |
|bmw          |7793    |31074.44 |
|mercedes-benz|6585    |19846.08 |
|subaru       |6379    |12154.55 |
|hyundai      |6253    |10131.6  |
|volkswagen   |5216    |10518.69 |
+-------------+--------+---------+


  6c. Fuel Type Distribution
+--------+--------+-----+
|fuel    |listings|pct  |
+--------+--------+-----+
|gas     |215757  |88.15|
|diesel  |16085   |6.57 |
|other   |7748    |3.1

2026-07-26 18:45:06 | INFO |   All analytical SQL queries completed.


+--------------+--------+-----+
|posting_period|listings|pct  |
+--------------+--------+-----+
|Afternoon     |105897  |43.26|
|Evening       |95559   |39.04|
|Night         |32533   |13.29|
|Morning       |10783   |4.41 |
+--------------+--------+-----+



In [0]:
# =============================================================================
# 7. EXPORT SINGLE CSV FOR TABLEAU
# =============================================================================

log.info("Exporting Featured Dataset as single CSV for Tableau...")
export_start = time.time()

df_export = spark.read.table(FEATURED_TABLE)
record_count_export = df_export.count()

df_export \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(EXPORT_VOLUME)

export_end  = time.time()
export_time = round(export_end - export_start, 2)

log.info(f"  Export path            : {EXPORT_VOLUME}")
log.info(f"  Records exported       : {record_count_export}")
log.info(f"  Export time            : {export_time}s")
log.info("  Export complete.")

2026-07-26 18:45:06 | INFO | Exporting Featured Dataset as single CSV for Tableau...
2026-07-26 18:45:20 | INFO |   Export path            : /Volumes/workspace/default/vehicle_data/vehicles_featured_tableau
2026-07-26 18:45:20 | INFO |   Records exported       : 244772
2026-07-26 18:45:20 | INFO |   Export time            : 13.81s
2026-07-26 18:45:20 | INFO |   Export complete.


In [0]:
# Read-back verification — confirms CSV is readable and record count matches
df_verify      = spark.read.option("header", True).csv(EXPORT_VOLUME)
verified_count = df_verify.count()
log.info(f"  Read-back record count : {verified_count}")

if verified_count == record_count_export:
    log.info("  ✓ Export verified — record count matches.")
else:
    log.info(f"  ⚠ MISMATCH — exported {record_count_export}, read back {verified_count}. Review required.")

2026-07-26 18:45:22 | INFO |   Read-back record count : 248460
2026-07-26 18:45:22 | INFO |   ⚠ MISMATCH — exported 244772, read back 248460. Review required.


In [0]:
log.info("")
log.info("  Pipeline Status        : SUCCESS")

2026-07-26 18:45:23 | INFO | 
2026-07-26 18:45:23 | INFO |   Pipeline Status        : SUCCESS


In [0]:
# =============================================================================
# 8. PIPELINE COMPLETION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)

log.info("")
log.info("\n" + "=" * 65)
log.info("  DELTA WAREHOUSE — COMPLETION REPORT")
log.info("=" * 65)
log.info(f"  Execution Time         : {execution_time}s")
log.info(f"  Records Loaded         : {record_count_in}")
log.info(f"  Records Exported (CSV) : {record_count_export}")
log.info(f"  Featured Delta Table   : {FEATURED_TABLE}")
log.info(f"  Tableau Export Path    : {EXPORT_VOLUME}")
log.info("")
log.info("  Steps completed:")
log.info("    ✓ Featured Delta table loaded")
log.info("    ✓ Table schema and metadata validated")
log.info("    ✓ Analytical SQL queries executed (10 queries)")
log.info("    ✓ Single CSV exported for Tableau")
log.info("  Pipeline Status        : SUCCESS")
log.info("")
log.info("  Batch pipeline complete.")
log.info("  Tableau can now connect to the exported CSV and build dashboards.")
log.info("=" * 65)

2026-07-26 18:45:23 | INFO | 
2026-07-26 18:45:23 | INFO | 
2026-07-26 18:45:23 | INFO |   DELTA WAREHOUSE — COMPLETION REPORT
2026-07-26 18:45:23 | INFO | =================================================================
2026-07-26 18:45:23 | INFO |   Execution Time         : 33.06s
2026-07-26 18:45:23 | INFO |   Records Loaded         : 244772
2026-07-26 18:45:23 | INFO |   Records Exported (CSV) : 244772
2026-07-26 18:45:23 | INFO |   Featured Delta Table   : workspace.default.featured_vehicles
2026-07-26 18:45:23 | INFO |   Tableau Export Path    : /Volumes/workspace/default/vehicle_data/vehicles_featured_tableau
2026-07-26 18:45:23 | INFO | 
2026-07-26 18:45:23 | INFO |   Steps completed:
2026-07-26 18:45:23 | INFO |     ✓ Featured Delta table loaded
2026-07-26 18:45:23 | INFO |     ✓ Table schema and metadata validated
2026-07-26 18:45:23 | INFO |     ✓ Analytical SQL queries executed (10 queries)
2026-07-26 18:45:23 | INFO |     ✓ Single CSV exported for Tableau
2026-07-26 18:45